In [1]:
import os
import openai


# Set API key directly
openai.api_key = "KEY"

# Verify the key is set
assert openai.api_key, "Please set OPENAI_API_KEY in your environment"
print("API key loaded!")

API key loaded!


In [2]:
import sentence_transformers
from sentence_transformers import SentenceTransformer

In [12]:
import os, json, pickle
from pathlib import Path


from typing import List, Dict, Tuple
from src.rag.retrieval import build_faiss_index, load_faiss_index, query_faiss_index

import faiss
from tqdm import tqdm
import numpy as np

In [4]:
# Cell 2 — Load FAISS index and metadata
VDB_DIR = Path("vectordb")
INDEX_PATH = VDB_DIR / "index.faiss"
DOCSTORE_PATH = VDB_DIR / "docstore.pkl"

# load FAISS index
index = faiss.read_index(str(INDEX_PATH))

# load metadata
with open(DOCSTORE_PATH, "rb") as f:
    docstore: List[Dict] = pickle.load(f)


print(f"Index nvecs: {index.ntotal}, metadata entries: {len(docstore)}")


Index nvecs: 15922, metadata entries: 15922


In [5]:
EMB_MODEL = "sentence-transformers/paraphrase-MiniLM-L3-v2"
sbert = SentenceTransformer(EMB_MODEL)
def retrieve_evidence(question: str, k: int = 5):
    """
    Given a free‐form question string, embed it (with your SBERT encoder) and
    query the FAISS index for the top‐k hits.
    """
    # example: you have your SBERT model loaded globally as `sbert`
    q_emb = sbert.encode(question, normalize_embeddings=True).tolist()
    hits = query_faiss_index(index, docstore, q_emb, top_k=k)
    # returns list of (metadata_record, distance)
    return hits

# Quick smoke test
hits = retrieve_evidence("What is Flublok?", k=2)
for rec, dist in hits:
    print(f"{rec['source_pdf'][:20]:20s} → {dist:.3f}   {rec.get('text','')[:80]}")


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Treanor et al. (2011 → 0.945   FluBlok,
Treanor et al. (2011 → 0.945   FluBlok,


In [13]:


MODEL       = "gpt-4"
MAX_STEPS   = 6
RETRIEVE_K  = 5

def generate_question(claim: str, history: list) -> str:
    msgs = [
        {"role":"system","content":"You are an expert fact‑checking assistant."},
        {"role":"user",  "content":f"Claim: “{claim}”\nGenerate one focused question to verify this claim."}
    ]
    if history:
        qa = "\n".join(f"Q: {h['q']}\nA: {h['a']}" for h in history)
        msgs.append({
            "role":"user",
            "content":f"Previous Q/A:\n{qa}\nNow generate the next focused question."
        })
    resp = openai.ChatCompletion.create(model=MODEL, messages=msgs, temperature=0.0)
    return resp.choices[0].message.content.strip()

def answer_question(question: str, evidence: List[Tuple[Dict, float]]) -> str:
    """
    Step 3: Answer the question using retrieved evidence.
    'evidence' is a list of (metadata_record, distance) tuples.
    """
    # Build a context string by unpacking each (rec, dist)
    ctx_lines = []
    for rec, dist in evidence:
        text = rec.get("text") or rec.get("ocr_text", "")
        snippet = text[:300].replace("\n", " ").strip()
        ctx_lines.append(f"- [{dist:.2f}] “{snippet}” ({rec['source_pdf']})")
    ctx = "\n".join(ctx_lines)

    messages = [
        {"role": "system", "content":
            "You are an expert fact‑checking assistant.\n"
            "Given a focused question and some context snippets, do the following:\n"
            "1) Restate the question briefly.\n"
            "2) Quote the key evidence.\n"
            "3) Draw a concise conclusion (yes/no/maybe) with one‐line reasoning.\n"
        },
        {"role": "user", "content":
            f"Question: {question}\n\n"
            f"Context snippets:\n{ctx}\n\n"
            "Answer in 1–2 sentences, quoting the most relevant fragment(s)."
        }
    ]

    resp = openai.ChatCompletion.create(
        model=MODEL,
        messages=messages,
        temperature=0.0
    )
    return resp.choices[0].message.content.strip()


def is_sufficient(claim: str, history: list) -> bool:
    qa = "\n".join(f"Q: {h['q']}\nA: {h['a']}" for h in history)
    msgs = [
        {"role":"system","content":"You are an expert fact‑checking assistant."},
        {"role":"user",  "content":f"Claim: “{claim}”\nCollected Q/A:\n{qa}\n\nIs this evidence sufficient? Answer yes or no."}
    ]
    resp = openai.ChatCompletion.create(model=MODEL, messages=msgs, temperature=0.0)
    return resp.choices[0].message.content.lower().startswith("y")


In [14]:

def corag_chain(claim: str) -> list:
    history = []
    for step in range(MAX_STEPS):
        q  = generate_question(claim, history)
        ev = retrieve_evidence(q, k=RETRIEVE_K)  # List[ (rec, dist) ]
        a  = answer_question(q, ev)

        # Re‑format evidence for JSON
        formatted_ev = [
            {"source": rec["source_pdf"], "text": rec.get("text",""), "score": dist}
            for rec, dist in ev
        ]

        history.append({
            "step":     step + 1,
            "q":        q,
            "evidence": formatted_ev,
            "a":        a
        })

        if is_sufficient(claim, history):
            break

    return history


# Smoke‐test on one claim
test = corag_chain("Flublok ensures identical antigenic match with WHO‑ and FDA‑selected flu strains.")
print(json.dumps(test, indent=2, ensure_ascii=False))

[
  {
    "step": 1,
    "q": "\"Does Flublok guarantee an identical antigenic match with the flu strains selected by the World Health Organization (WHO) and the U.S. Food and Drug Administration (FDA)?\"",
    "evidence": [
      {
        "source": "FlublokPI",
        "text": "of VE of Flublok against all strains, regardless of antigenic match, isolated from any",
        "score": 0.7007983922958374
      },
      {
        "source": "FlublokPI",
        "text": "Flublok is standardized according to United States Public Health Service (USPHS)",
        "score": 0.6621873378753662
      },
      {
        "source": "FlublokPI",
        "text": "The efficacy of Flublok Quadrivalent is relevant to Flublok because both vaccines are",
        "score": 0.6532954573631287
      },
      {
        "source": "Treanor et al. (2011)",
        "text": "(FluBlok®).",
        "score": 0.649735152721405
      },
      {
        "source": "FlublokPI",
        "text": "[14]). Data for Flublok Quadri

In [15]:
CLAIMS_PATH = Path("data/Flublok_Claims.json")
with CLAIMS_PATH.open() as f:
    claims_list = json.load(f)["claims"]

output = {"claims": []}
for item in tqdm(claims_list, desc="Running CoRAG"):
    steps = corag_chain(item["claim"])
    output["claims"].append({
        "claim": item["claim"],
        "corag_steps": steps
    })

# write to corag.json
with open("corag(Update).json","w", encoding="utf-8") as wf:
    json.dump(output, wf, indent=2, ensure_ascii=False)

print("✅ CoRAG results saved to corag(Update).json")

Running CoRAG: 100%|██████████| 9/9 [03:16<00:00, 21.88s/it]

✅ CoRAG results saved to corag(Update).json
